In [ ]:



# CELL 1 — IMPORTS

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

print("PyTorch version:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


# CELL 2 — CONFIG (change your hyperparameters here)

MAX_LEN    = 15      # max words per sentence
EMBED_DIM  = 50      # word vector size
HIDDEN     = 64      # RNN hidden state size
LAYERS     = 1       # number of RNN layers
BIDIR      = False   # bidirectional? True or False
DROPOUT    = 0.3
BATCH_SIZE = 2
EPOCHS     = 10
LR         = 0.001

PyTorch version: 2.10.0+cu128
Device: cuda


In [ ]:
# CELL 3 — DATA
train_texts = [
    "i love this movie it is amazing",
    "this film was absolutely terrible",
    "wonderful performance loved every minute",
    "worst film i have ever seen",
    "brilliant storyline and great acting",
    "boring and completely predictable",
]
train_labels = [1, 0, 1, 0, 1, 0]   # 1=positive, 0=negative

val_texts  = ["really enjoyed this film", "hated every second of it"]
val_labels = [1, 0]

print(f"Train samples : {len(train_texts)}")
print(f"Val samples   : {len(val_texts)}")


Train samples : 6
Val samples   : 2


In [ ]:
# CELL 4 — VOCABULARY BUILDER
def build_vocab(texts):
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for text in texts:
        for word in text.lower().split():
            if word not in vocab:
                vocab[word] = len(vocab)
    return vocab

def encode(text, vocab, max_len):
    words  = text.lower().split()
    indices = [vocab.get(w, 1) for w in words]       # 1 = <UNK>
    indices = indices + [0] * (max_len - len(indices)) # pad
    return indices[:max_len]                           # truncate

vocab = build_vocab(train_texts + val_texts)
print(f"Vocabulary size: {len(vocab)}")
print(f"Sample encoding: {encode('i love this movie', vocab, MAX_LEN)}")

# CELL 5 — DATASET & DATALOADER
class TextDataset(Dataset):

    def __init__(self, texts, labels, vocab, max_len):
        self.data = [
            torch.tensor(encode(t, vocab, max_len), dtype=torch.long)
            for t in texts
        ]
        self.labels = [torch.tensor(l, dtype=torch.long) for l in labels]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]


train_dataset = TextDataset(train_texts, train_labels, vocab, MAX_LEN)
val_dataset   = TextDataset(val_texts,   val_labels,   vocab, MAX_LEN)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

# Quick check
sample_x, sample_y = train_dataset[0]
print(f"Sample input shape : {sample_x.shape}")
print(f"Sample label       : {sample_y}")



Vocabulary size: 35
Sample encoding: [2, 3, 4, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Sample input shape : torch.Size([15])
Sample label       : 1


In [ ]:
# CELL 6 — MODEL
class SentimentRNN(nn.Module):

    def __init__(self, vocab_size, embed_dim, hidden_size, num_classes,
        num_layers=1, bidirectional=False, dropout=0.3):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        self.rnn = nn.RNN(
            input_size    = embed_dim,
            hidden_size   = hidden_size,
            num_layers    = num_layers,
            batch_first   = True,
            bidirectional = bidirectional,
            dropout       = dropout if num_layers > 1 else 0
        )

        self.dropout = nn.Dropout(dropout)

        fc_input = hidden_size * 2 if bidirectional else hidden_size
        self.fc  = nn.Linear(fc_input, num_classes)

    def forward(self, x):
        # x → (batch, seq_len)
        embedded = self.dropout(self.embedding(x))
        # embedded → (batch, seq_len, embed_dim)

        output, hidden = self.rnn(embedded)
        # hidden → (num_layers * directions, batch, hidden_size)

        last = self.dropout(hidden[-1])
        # last → (batch, hidden_size)

        return self.fc(last)
        # return → (batch, num_classes)


model = SentimentRNN(
    vocab_size    = len(vocab),
    embed_dim     = EMBED_DIM,
    hidden_size   = HIDDEN,
    num_classes   = 2,
    num_layers    = LAYERS,
    bidirectional = BIDIR,
    dropout       = DROPOUT
).to(device)

print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")



SentimentRNN(
  (embedding): Embedding(35, 50, padding_idx=0)
  (rnn): RNN(50, 64, batch_first=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=64, out_features=2, bias=True)
)

Total parameters: 9,304


In [ ]:
# CELL 7 — LOSS & OPTIMIZER
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

print("Loss     :", criterion)
print("Optimizer:", optimizer)



Loss     : CrossEntropyLoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [ ]:
# CELL 8 — TRAIN & EVALUATE FUNCTIONS
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct = 0, 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        predictions = model(X_batch)
        loss        = criterion(predictions, y_batch)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        correct    += (predictions.argmax(1) == y_batch).sum().item()

    avg_loss = total_loss / len(loader)
    accuracy = correct   / len(loader.dataset)
    return avg_loss, accuracy


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0, 0

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            predictions = model(X_batch)
            loss        = criterion(predictions, y_batch)

            total_loss += loss.item()
            correct    += (predictions.argmax(1) == y_batch).sum().item()

    avg_loss = total_loss / len(loader)
    accuracy = correct   / len(loader.dataset)
    return avg_loss, accuracy

print("Functions ready.")



Functions ready.


In [ ]:
# CELL 9 — TRAINING LOOP
best_val_acc = 0.0

for epoch in range(EPOCHS):

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss,   val_acc   = evaluate(model, val_loader, criterion)

    print(f"Epoch {epoch+1:02d}/{EPOCHS}  |  "
          f"Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.2%}  |  "
          f"Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.2%}")

    # Save best model weights
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_rnn.pt")
        print(f"  ✓ Best model saved (Val Acc: {best_val_acc:.2%})")

print(f"\nTraining complete. Best Val Accuracy: {best_val_acc:.2%}")




Epoch 01/10  |  Train Loss: 0.6981  Train Acc: 50.00%  |  Val Loss: 0.6933  Val Acc: 50.00%
  ✓ Best model saved (Val Acc: 50.00%)
Epoch 02/10  |  Train Loss: 0.7038  Train Acc: 50.00%  |  Val Loss: 0.6933  Val Acc: 50.00%
Epoch 03/10  |  Train Loss: 0.7211  Train Acc: 0.00%  |  Val Loss: 0.6933  Val Acc: 50.00%
Epoch 04/10  |  Train Loss: 0.6954  Train Acc: 33.33%  |  Val Loss: 0.6933  Val Acc: 50.00%
Epoch 05/10  |  Train Loss: 0.6827  Train Acc: 66.67%  |  Val Loss: 0.6935  Val Acc: 50.00%
Epoch 06/10  |  Train Loss: 0.6910  Train Acc: 50.00%  |  Val Loss: 0.6936  Val Acc: 50.00%
Epoch 07/10  |  Train Loss: 0.6662  Train Acc: 100.00%  |  Val Loss: 0.6937  Val Acc: 50.00%
Epoch 08/10  |  Train Loss: 0.6844  Train Acc: 83.33%  |  Val Loss: 0.6938  Val Acc: 50.00%
Epoch 09/10  |  Train Loss: 0.7004  Train Acc: 33.33%  |  Val Loss: 0.6941  Val Acc: 50.00%
Epoch 10/10  |  Train Loss: 0.6869  Train Acc: 66.67%  |  Val Loss: 0.6944  Val Acc: 50.00%

Training complete. Best Val Accuracy: 50

In [ ]:
# CELL 10 — PREDICT ON NEW TEXT
def predict(text, model, vocab, max_len):
    model.eval()
    encoded = encode(text, vocab, max_len)
    x       = torch.tensor([encoded], dtype=torch.long).to(device)

    with torch.no_grad():
        output = model(x)
        probs  = torch.softmax(output, dim=1)
        pred   = output.argmax(1).item()

    label      = "Positive 😊" if pred == 1 else "Negative 😞"
    confidence = probs[0][pred].item()

    print(f"Text       : {text}")
    print(f"Prediction : {label}  ({confidence:.2%} confidence)")
    print()


# Test it
predict("this movie was absolutely wonderful",  model, vocab, MAX_LEN)
predict("i  hated this boring film",   model, vocab, MAX_LEN)
predict("great acting and brilliant direction",  model, vocab, MAX_LEN)

Text       : this movie was absolutely wonderful
Prediction : Positive 😊  (50.05% confidence)

Text       : i  hated this boring film
Prediction : Negative 😞  (50.05% confidence)

Text       : great acting and brilliant direction
Prediction : Positive 😊  (50.14% confidence)

